# Session 5 · Storytelling with Generative AI

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-05_story_arc.py`

In [1]:
"""
Session 5 · Example 1 — Plan a story's structure before asking AI to write it.

Most stories follow a shape. The classic five-stage arc (exposition, rising
action, climax, falling action, resolution) can be given a share of the total
length. Asking the AI for "a 1,500-word story" gives a saggy middle; asking
for each stage with a word budget and a tension level gives a planned arc.

Run:  python session-05_story_arc.py
Creates: session-05_tension_curve.png   (matplotlib)
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TOTAL_WORDS = 1500
ARC = [  # stage, share of words, tension at the END of the stage (0-10), what happens
    ("Exposition",     0.15, 2, "Meet Ayesha, a CS student who builds a homework-helper bot"),
    ("Rising action",  0.40, 7, "The bot starts writing her friends' assignments; a teacher grows suspicious"),
    ("Climax",         0.15, 10, "Ayesha must choose: shut the bot down or let a friend take the blame"),
    ("Falling action", 0.20, 5, "She confesses and redesigns the bot to give hints, not answers"),
    ("Resolution",     0.10, 3, "The tutor-bot wins the campus innovation prize"),
]

start, points = 0, [(0, 1)]
print(f"{'Stage':15}{'Words':>6}{'Range':>13}{'Tension':>8}  Beat")
for stage, share, tension, beat in ARC:
    words = round(TOTAL_WORDS * share)
    print(f"{stage:15}{words:6}{start:>6}-{start + words:<6}{tension:>8}  {beat}")
    start += words
    points.append((start, tension))
print(f"Total: {start} words")

xs, ys = zip(*points)
plt.figure(figsize=(8, 3.5))
plt.plot(xs, ys, marker="o", color="#FF6B1A", linewidth=2)
for (stage, *_), x, y in zip(ARC, xs[1:], ys[1:]):
    plt.annotate(stage, (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
plt.ylim(0, 11.5)
plt.xlabel("Word position in the story")
plt.ylabel("Tension (0-10)")
plt.title("Planned tension curve")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("session-05_tension_curve.png", dpi=80)
print("Saved -> session-05_tension_curve.png")

Stage           Words        Range Tension  Beat
Exposition        225     0-225          2  Meet Ayesha, a CS student who builds a homework-helper bot
Rising action     600   225-825          7  The bot starts writing her friends' assignments; a teacher grows suspicious
Climax            225   825-1050        10  Ayesha must choose: shut the bot down or let a friend take the blame
Falling action    300  1050-1350         5  She confesses and redesigns the bot to give hints, not answers
Resolution        150  1350-1500         3  The tutor-bot wins the campus innovation prize
Total: 1500 words
Saved -> session-05_tension_curve.png


### Example · `session-05_character_sheet.py`

In [2]:
"""
Session 5 · Example 2 — A character bible that keeps AI stories consistent.

Problem: over a long chat the AI "forgets" that the hero has green eyes or
hates coffee. Fix: write each character once as structured data and paste
the same compact "bible" at the top of every chapter prompt.

Run:  python session-05_character_sheet.py
"""
from dataclasses import dataclass, asdict


@dataclass
class Character:
    name: str
    role: str          # protagonist, antagonist, mentor, ...
    age: int
    want: str          # what they consciously chase
    need: str          # what they actually need to learn (drives the arc)
    flaw: str
    voice: str         # how they speak
    details: str       # fixed physical or personal facts

    def card(self) -> str:
        return (f"{self.name} ({self.role}, {self.age}): wants {self.want}; needs {self.need}; "
                f"flaw: {self.flaw}; speaks {self.voice}; fixed facts: {self.details}.")


cast = [
    Character("Ayesha", "protagonist", 19, "to be the best coder in class",
              "to accept help and give credit", "pride", "fast, with tech slang",
              "wears round glasses, drinks chai, never coffee"),
    Character("Sir Kamran", "mentor", 45, "students who think for themselves",
              "to trust that students can change", "stubborn", "calmly, answers questions with questions",
              "grey beard, always carries a red marker"),
    Character("Bilal", "friend / foil", 20, "to pass without effort",
              "to own his mistakes", "laziness", "in jokes to dodge serious talk",
              "plays cricket for the university team"),
]

bible = "STORY BIBLE (keep these facts consistent):\n" + "\n".join("- " + c.card() for c in cast)
print(bible)
print("\nChapter prompt template:")
print(bible.splitlines()[0] + " [paste bible]\nWrite Chapter 3 (about 400 words): "
      "Sir Kamran notices two identical assignments. Stay in Ayesha's point of view.")
print("\nFields stored per character:", list(asdict(cast[0]).keys()))

STORY BIBLE (keep these facts consistent):
- Ayesha (protagonist, 19): wants to be the best coder in class; needs to accept help and give credit; flaw: pride; speaks fast, with tech slang; fixed facts: wears round glasses, drinks chai, never coffee.
- Sir Kamran (mentor, 45): wants students who think for themselves; needs to trust that students can change; flaw: stubborn; speaks calmly, answers questions with questions; fixed facts: grey beard, always carries a red marker.
- Bilal (friend / foil, 20): wants to pass without effort; needs to own his mistakes; flaw: laziness; speaks in jokes to dodge serious talk; fixed facts: plays cricket for the university team.

Chapter prompt template:
STORY BIBLE (keep these facts consistent): [paste bible]
Write Chapter 3 (about 400 words): Sir Kamran notices two identical assignments. Stay in Ayesha's point of view.

Fields stored per character: ['name', 'role', 'age', 'want', 'need', 'flaw', 'voice', 'details']


### Example · `session-05_branching_story.py`

In [3]:
"""
Session 5 · Example 3 — Check an interactive (choose-your-own-path) story.

Interactive stories are graphs: each scene is a node, each choice an edge.
When an AI writes branches, it often leaves dead ends (a choice that points
to a scene that does not exist) or scenes nobody can reach. A breadth-first
search finds every reachable scene, every ending and every broken link.

Run:  python session-05_branching_story.py
"""
from collections import deque

story = {
    "start":       {"text": "Your bot can now write full assignments.",
                    "choices": {"Sell it to classmates": "sell", "Turn it into a tutor": "tutor"}},
    "sell":        {"text": "Money rolls in, then a teacher calls you in.",
                    "choices": {"Confess": "confess", "Deny everything": "deny"}},
    "tutor":       {"text": "The tutor-bot gives hints only. Grades go up.",
                    "choices": {"Enter the innovation fair": "fair_win"}},
    "confess":     {"text": "You lose marks but keep your integrity.", "choices": {}},
    "deny":        {"text": "The logs prove it. Suspension.", "choices": {"Appeal": "appeal"}},  # broken
    "fair_win":    {"text": "You win the fair and a summer internship.", "choices": {}},
    "secret_room": {"text": "A scene the writer forgot to link.", "choices": {}},
}


def analyse(graph, start="start"):
    seen, queue, broken, endings = {start}, deque([(start, [start])]), [], []
    paths = {}
    while queue:
        node, path = queue.popleft()
        choices = graph[node]["choices"]
        if not choices:
            endings.append(node)
            paths[node] = path
        for label, target in choices.items():
            if target not in graph:
                broken.append((node, label, target))
            elif target not in seen:
                seen.add(target)
                queue.append((target, path + [target]))
    unreachable = sorted(set(graph) - seen)
    return endings, broken, unreachable, paths


endings, broken, unreachable, paths = analyse(story)
print("Endings found:", endings)
for e in endings:
    print(f"  shortest path to '{e}': {' -> '.join(paths[e])}")
print("Broken links:", [f"{n} --'{l}'--> {t}" for n, l, t in broken])
print("Unreachable scenes:", unreachable)
print("Story is", "VALID" if not broken and not unreachable else "NOT valid yet: fix the items above")

Endings found: ['confess', 'fair_win']
  shortest path to 'confess': start -> sell -> confess
  shortest path to 'fair_win': start -> tutor -> fair_win
Broken links: ["deny --'Appeal'--> appeal"]
Unreachable scenes: ['secret_room']
Story is NOT valid yet: fix the items above


### Example · `session-05_project_story_bible.py`

In [4]:
"""
Session 5 · Mini project (full solution) — Story bible + continuity checker.

Real-life problem: you are co-writing a 5-chapter story with an AI for a
children's reading club. Across chapters the AI drifts: names change
spelling, a character's fixed facts change, and banned words for the age
group slip in. This tool:

  1. builds a chapter-by-chapter prompt plan from an arc,
  2. scans each generated chapter against the story bible,
  3. reports every continuity or suitability problem with its chapter number.

Run:  python session-05_project_story_bible.py
"""
import re

BIBLE = {
    "names": {"Zara": ["Zahra", "Sara"], "Moti": ["Motee", "Mothi"]},   # canonical: known wrong variants
    "facts": {"Moti": ("orange cat", ["black cat", "grey cat", "dog"])},  # fixed fact and contradictions
    "banned": ["stupid", "kill", "hate"],                                  # reading-club age rule
}
ARC = ["Zara finds a lost orange cat called Moti",
       "Moti leads Zara to a hidden garden on the school roof",
       "The garden is dying because the water tank leaks",
       "Zara and her class build a rainwater collector",
       "The garden blooms and Moti becomes the school mascot"]

print("PROMPT PLAN")
for i, beat in enumerate(ARC, 1):
    print(f"  Ch{i}: Using the story bible, write ~300 words for ages 7-9: {beat}.")

chapters = {
    1: "Zara found a small orange cat behind the canteen. She named him Moti.",
    2: "Moti ran up the stairs. Zahra followed him to a garden full of plants.",
    3: "The plants looked sad. 'I hate leaks,' said Zara. Moti, the grey cat, sniffed the tank.",
    4: "The class built a collector from old bottles. Zara was proud.",
    5: "Flowers bloomed everywhere and Motee became the mascot.",
}


def check(chapter_no: int, text: str) -> list:
    issues = []
    for canon, variants in BIBLE["names"].items():
        for v in variants:
            if re.search(rf"\b{v}\b", text):
                issues.append(f"name drift: '{v}' should be '{canon}'")
    for who, (fact, contradictions) in BIBLE["facts"].items():
        for c in contradictions:
            if c in text.lower():
                issues.append(f"fact clash: {who} is an {fact}, text says '{c}'")
    for word in BIBLE["banned"]:
        if re.search(rf"\b{word}\b", text, re.I):
            issues.append(f"age rule: avoid the word '{word}'")
    return issues


print("\nCONTINUITY REPORT")
total = 0
for no, text in chapters.items():
    issues = check(no, text)
    total += len(issues)
    print(f"  Ch{no}: " + ("OK" if not issues else " | ".join(issues)))
print(f"\n{total} issue(s) found. Send each flagged chapter back with the bible and the exact fix.")

PROMPT PLAN
  Ch1: Using the story bible, write ~300 words for ages 7-9: Zara finds a lost orange cat called Moti.
  Ch2: Using the story bible, write ~300 words for ages 7-9: Moti leads Zara to a hidden garden on the school roof.
  Ch3: Using the story bible, write ~300 words for ages 7-9: The garden is dying because the water tank leaks.
  Ch4: Using the story bible, write ~300 words for ages 7-9: Zara and her class build a rainwater collector.
  Ch5: Using the story bible, write ~300 words for ages 7-9: The garden blooms and Moti becomes the school mascot.

CONTINUITY REPORT
  Ch1: OK
  Ch2: name drift: 'Zahra' should be 'Zara'
  Ch3: fact clash: Moti is an orange cat, text says 'grey cat' | age rule: avoid the word 'hate'
  Ch4: OK
  Ch5: name drift: 'Motee' should be 'Moti'

4 issue(s) found. Send each flagged chapter back with the bible and the exact fix.
